# Notebook 05: Full Pipeline & Lead Generation

Notebook ini mendemonstrasikan end-to-end pipeline chatbot AI untuk layanan Nasi Kotak Catering.

Pipeline menggabungkan:
1. **Conversation Manager**: State/memory user.
2. **RAG Service**: Retrieval konteks produk dan knowledge base.
3. **LLM Service (Groq)**: Prompting, intent recognition, entity extraction dengan schema JSON.
4. **Sales Engine**: Logika rekomendasi, penentuan `purchase_intent`, upsell, dan cross-sell.
5. **Lead Manager**: Trigger penangkapan prospek (lead) ketika intent memuncak dan pembuatan CTA link WhatsApp.

Semua orkestrasi di atas dibungkus dalam `src.pipeline.ChatPipeline`.

## 5.1 Setup & Inisialisasi

In [1]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.database import init_db, SessionLocal, Lead
from src.pipeline import ChatPipeline
from src.config import PROJECT_ROOT

# Inisialisasi Database (SQLite)
init_db()
db = SessionLocal()

# Inisialisasi Full Pipeline
pipeline = ChatPipeline()
print("✅ Pipeline siap.")

Database initialized successfully.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Berhasil memuat index FAISS dengan 22 vektor.
✅ Pipeline siap.


--- 
## 5.2 Helper Function untuk Menampilkan Hasil Chat

In [2]:
import textwrap

def print_chat_result(user_msg, result):
    print("=" * 80)
    print(textwrap.fill(f"User: {user_msg}", 80))
    print("-" * 80)
    
    if "error" in result:
        print(f"❌ Error: {result.get('error')}")
        return
        
    print(textwrap.fill(f"Bot: {result.get('reply', '')}", 80))
    print("-" * 80)
    print(f"📌 Intent: {result.get('intent', 'N/A')}")
    print(f"🔥 Purchase Intent: {result.get('purchase_intent', 'N/A')}")
    
    entities = result.get('entities', {})
    if any(v is not None for v in entities.values()):
        print(f"📦 Extracted Entities:")
        for k, v in entities.items():
            if v is not None:
                print(f"   - {k}: {v}")
                
    if result.get('actions'):
        print(f"⚙️ Actions: {result.get('actions')}")
        
    if result.get('needs_handover'):
        print(f"⚠️ Handover Triggered: {result.get('handover_reason')}")
        
    if result.get('lead_status') == 'captured':
        print(f"✅ LEAD CAPTURED! (ID: {result.get('lead_id')})")
        if result.get('whatsapp_link'):
            print(f"🔗 WhatsApp CTA: {result.get('whatsapp_link')}")
    print("=" * 80)
    print()

--- 
## 5.3 Scenario A: Customer Individu (Arisan, Budget 25rb)
Skenario ini akan menunjukkan bagaimana `purchase_intent` memuncak dan Lead di-capture.

In [3]:
session_a = "demo_session_A"
scenario_a_messages = [
    "Halo, saya mau cari nasi kotak untuk arisan bulan depan.",
    "Budgetnya sekitar 25 ribu per box.",
    "Butuh 50 box untuk tanggal 15 Oktober.",
    "Oke saya mau pesan paket Ayam Kampung itu."
]

for msg in scenario_a_messages:
    result = pipeline.chat(msg, session_id=session_a, db=db)
    print_chat_result(msg, result)

User: Halo, saya mau cari nasi kotak untuk arisan bulan depan.
--------------------------------------------------------------------------------
Bot: Halo kak! Saya senang membantu cari nasi kotak untuk arisan bulan depan.
Apakah acara ini gathering atau meeting?
--------------------------------------------------------------------------------
📌 Intent: greeting
🔥 Purchase Intent: LOW
📦 Extracted Entities:
   - session_id: demo_session_A
   - purchase_intent: LOW
   - messages: [{'sender': 'user', 'text': 'Halo, saya mau cari nasi kotak untuk arisan bulan depan.'}, {'sender': 'bot', 'text': 'Halo kak! Saya senang membantu cari nasi kotak untuk arisan bulan depan. Apakah acara ini gathering atau meeting?'}]

User: Budgetnya sekitar 25 ribu per box.
--------------------------------------------------------------------------------
Bot: Halo kak! Saya senang membantu cari nasi kotak untuk arisan bulan depan.
Apakah acara ini gathering atau meeting?
--------------------------------------------

--- 
## 5.4 Scenario B: Corporate (Meeting 100 Box, Web Redirect)
Skenario ketika intent = ordering, bot akan direct ke URL website.

In [4]:
session_b = "demo_session_B"
scenario_b_messages = [
    "Butuh katering meeting kantor 100 orang, budget 30rb, apa yang cocok?",
    "Paket Broiler Jumbo dan tambahan snack box standar sepertinya pas. Saya ambil itu.",
    "Kirim ke Sudirman hari Jumat jam 12, tolong diproses sekarang."
]

for msg in scenario_b_messages:
    result = pipeline.chat(msg, session_id=session_b, db=db)
    print_chat_result(msg, result)

[INFO] TPM limit tercapai. Menunggu 17.8s...
User: Butuh katering meeting kantor 100 orang, budget 30rb, apa yang cocok?
--------------------------------------------------------------------------------
Bot: Halo kak! Kamu butuh katering meeting kantor 100 orang dengan budget 30rb.
Saya rekomendasikan Nasi Kotak Broiler, cocok untuk acara meeting kantor dengan
harga terjangkau! Harga Rp 20.000 / box, minimum order 20 box.
--------------------------------------------------------------------------------
📌 Intent: recommendation
🔥 Purchase Intent: LOW
📦 Extracted Entities:
   - session_id: demo_session_B
   - quantity: 100
   - budget_per_box: 30000.0
   - event_type: meeting
   - purchase_intent: LOW
   - messages: [{'sender': 'user', 'text': 'Butuh katering meeting kantor 100 orang, budget 30rb, apa yang cocok?'}, {'sender': 'bot', 'text': 'Halo kak! Kamu butuh katering meeting kantor 100 orang dengan budget 30rb. Saya rekomendasikan Nasi Kotak Broiler, cocok untuk acara meeting kantor d

--- 
## 5.5 Scenario C: Event Organizer (Custom Menu / Handoff)
Skenario ketika user memiliki kebutuhan dietary yang rumit dan memaksa fallback safety-net `needs_handover`.

In [5]:
session_c = "demo_session_C"
scenario_c_messages = [
    "Halo, saya EO. Mau pesan 80 box untuk besok.",
    "Saya butuh 10 porsi menu khusus vegan strict, bisa dijamin bebas kontaminasi produk hewani dari dapur?"
]

for msg in scenario_c_messages:
    result = pipeline.chat(msg, session_id=session_c, db=db)
    print_chat_result(msg, result)

[INFO] TPM limit tercapai. Menunggu 41.7s...
User: Halo, saya EO. Mau pesan 80 box untuk besok.
--------------------------------------------------------------------------------
Bot: Halo EO, boleh dibantu ya? 🙏
--------------------------------------------------------------------------------
📌 Intent: greeting
🔥 Purchase Intent: LOW
📦 Extracted Entities:
   - session_id: demo_session_C
   - quantity: 80
   - event_date: besok
   - purchase_intent: LOW
   - messages: [{'sender': 'user', 'text': 'Halo, saya EO. Mau pesan 80 box untuk besok.'}, {'sender': 'bot', 'text': 'Halo EO, boleh dibantu ya? 🙏'}]
⚙️ Actions: ['tanya_event_type', 'tanya_budget_per_box']

[INFO] TPM limit tercapai. Menunggu 16.0s...
User: Saya butuh 10 porsi menu khusus vegan strict, bisa dijamin bebas
kontaminasi produk hewani dari dapur?
--------------------------------------------------------------------------------
Bot: Untuk permintaan ini, saya mau pastikan dulu dengan tim kami ya kak, biar
nggak salah info 🙏  Un

--- 
## 5.6 Scenario D: Anti-Hallucination
Skenario ketika user menanyakan produk yang jelas tidak ada di knowledge base (misal: sushi).

In [6]:
session_d = "demo_session_D"
result = pipeline.chat("Halo, ada menu sushi box nggak?", session_id=session_d, db=db)
print_chat_result("Halo, ada menu sushi box nggak?", result)

[INFO] TPM limit tercapai. Menunggu 42.7s...
User: Halo, ada menu sushi box nggak?
--------------------------------------------------------------------------------
Bot: Halo kak! Kami tidak memiliki menu sushi box. Namun, kami dapat menawarkan
Snack Box sebagai alternatif. Apakah Anda ingin tahu lebih lanjut tentang Snack
Box?
--------------------------------------------------------------------------------
📌 Intent: product_inquiry
🔥 Purchase Intent: LOW
📦 Extracted Entities:
   - session_id: demo_session_D
   - event_type: meeting
   - purchase_intent: LOW
   - messages: [{'sender': 'user', 'text': 'Halo, ada menu sushi box nggak?'}, {'sender': 'bot', 'text': 'Halo kak! Kami tidak memiliki menu sushi box. Namun, kami dapat menawarkan Snack Box sebagai alternatif. Apakah Anda ingin tahu lebih lanjut tentang Snack Box?'}]
⚙️ Actions: ['tawarkan_snack_box']



--- 
## 5.7 Lead Report
Mari kita query SQLite database untuk melihat Lead yang berhasil di-capture selama simulasi di atas.

In [7]:
import pandas as pd

leads = pipeline.lead_manager.get_all_leads(db)
lead_data = []
for l in leads:
    lead_data.append({
        "ID": l.id,
        "Event": l.event_type,
        "Qty": l.quantity,
        "Budget": l.budget,
        "Location": l.location,
        "Purchase Intent": l.purchase_intent,
        "Created": l.created_at
    })

df_leads = pd.DataFrame(lead_data)
df_leads

""
